# F1 2026 Runner Up Title Odds

# Data Preparation
## Imports

In [1]:
import pandas as pd
import numpy as np

## Load Tracker

In [2]:
raw = pd.read_excel('F1_2026_Title_Tracker_v2.xlsx', sheet_name='Weekly_Points_2026', header=None)
print(raw.shape)
print(raw.iloc[0, 0:5].tolist())
print(raw.iloc[1, 0:8].tolist())

(10, 27)
['Race_Number', 'Race_Name', 'Sprint_Weekend', 'K. Antonelli', nan]
[nan, nan, nan, 'Finish_Position', 'Points_Scored', 'Cumulative_Points', 'Gap_to_Leader', 'Finish_Position']


## Reshape to Long Format

In [3]:
drivers = [raw.iloc[0, 3 + 4*k] for k in range(6)]

frames = []
for k, driver in enumerate(drivers):
    start = 3 + 4*k
    block = raw.iloc[3:, [0, 1, 2, start, start + 1]].copy()
    block.columns = ['Race_Number', 'Race_Name', 'Sprint_Weekend', 'Finish_Position', 'Points_Scored']
    block.insert(0, 'Driver', driver)
    frames.append(block)

results = pd.concat(frames, ignore_index=True)
print(results.shape)
results.head(8)

(42, 6)


,Driver,Race_Number,Race_Name,Sprint_Weekend,Finish_Position,Points_Scored
0,K. Antonelli,17,Singapore GP,Yes,3,20
1,K. Antonelli,18,United States GP,No,1,25
2,K. Antonelli,19,Gran Premio De Mexico,No,3,15
3,K. Antonelli,20,Sao Paula GP,No,5,10
4,K. Antonelli,21,Las Vegas GP,No,4,12
5,K. Antonelli,22,Qatar GP,No,NaN,NaN
6,K. Antonelli,23,Abu Dhabi GP,No,NaN,NaN
7,G. Russell,17,Singapore GP,Yes,9,2


In [4]:
results['Race_Number'] = pd.to_numeric(results['Race_Number'])
results['Points_Scored'] = pd.to_numeric(results['Points_Scored'], errors='coerce')
results['Finish_Num'] = pd.to_numeric(results['Finish_Position'], errors='coerce')

print(results.dtypes)
print(results[['Points_Scored', 'Finish_Num']].isna().sum())

Driver                 str
Race_Number          int64
Race_Name              str
Sprint_Weekend         str
Finish_Position     object
Points_Scored      float64
Finish_Num         float64
dtype: object
Points_Scored    12
Finish_Num       12
dtype: int64


## Check Against Official Standings

In [5]:
baseline = pd.Series({raw.iloc[0, 3 + 4*k]: raw.iloc[2, 4 + 4*k] for k in range(6)}).astype(float)
print(baseline)

K. Antonelli     320.0
G. Russell       236.0
L. Hamilton      214.0
C. Leclerc       191.0
L. Norris        188.0
M. Verstappen    188.0
dtype: float64


In [7]:
official = pd.read_excel('F1_2026_Title_Tracker_v2.xlsx', sheet_name='Title_Race',
                         usecols='A,E', nrows=6).set_index('Driver')['Official_Latest']

current = baseline + results.groupby('Driver')['Points_Scored'].sum()

check = pd.DataFrame({'Baseline': baseline, 'Computed': current, 'Official': official})
check['Match'] = check['Computed'] == check['Official']
check

,Baseline,Computed,Official,Match
C. Leclerc,191.0,250.0,250,True
G. Russell,236.0,264.0,264,True
K. Antonelli,320.0,402.0,402,True
L. Hamilton,214.0,267.0,267,True
L. Norris,188.0,264.0,264,True
M. Verstappen,188.0,304.0,304,True


## Current Standings (P2 Contenders)

In [8]:
contenders = ['M. Verstappen', 'L. Hamilton', 'G. Russell', 'C. Leclerc', 'L. Norris']

standings = current[contenders].sort_values(ascending=False).rename('Points').to_frame()
standings['Gap_to_Leader'] = standings['Points'] - standings['Points'].max()
standings

,Points,Gap_to_Leader
M. Verstappen,304.0,0.0
L. Hamilton,267.0,-37.0
G. Russell,264.0,-40.0
L. Norris,264.0,-40.0
C. Leclerc,250.0,-54.0


## Races Left and Points Available

In [9]:
played = results.groupby('Race_Number')['Points_Scored'].apply(lambda s: s.notna().all())
remaining = results.drop_duplicates('Race_Number').loc[lambda d: ~d['Race_Number'].map(played), ['Race_Number', 'Race_Name', 'Sprint_Weekend']]
remaining['Max_Points'] = np.where(remaining['Sprint_Weekend'] == 'Yes', 33, 25)

print(len(remaining), remaining['Max_Points'].sum())
remaining

2 50


,Race_Number,Race_Name,Sprint_Weekend,Max_Points
5,22,Qatar GP,No,25
6,23,Abu Dhabi GP,No,25


## Load Races 1-16

In [10]:
hist_raw = pd.read_excel('F1_2026_Championship_Data.xlsx', sheet_name='Race_Results_1_16', header=None)
print(hist_raw.shape)
print(hist_raw.iloc[0, 0:5].tolist())
print(hist_raw.iloc[1, 0:5].tolist())

(18, 15)
['Race_Number', 'Race_Name', 'Sprint_Weekend', 'K. Antonelli', nan]
[nan, nan, nan, 'Finish_Position', 'Points_Scored']


In [11]:
hist_drivers = [hist_raw.iloc[0, 3 + 2*k] for k in range(6)]

hist_frames = []
for k, driver in enumerate(hist_drivers):
    start = 3 + 2*k
    block = hist_raw.iloc[2:, [0, 1, 2, start, start + 1]].copy()
    block.columns = ['Race_Number', 'Race_Name', 'Sprint_Weekend', 'Finish_Position', 'Points_Scored']
    block.insert(0, 'Driver', driver)
    hist_frames.append(block)

history = pd.concat(hist_frames, ignore_index=True)
print(history.shape)
history.head(8)

(96, 6)


,Driver,Race_Number,Race_Name,Sprint_Weekend,Finish_Position,Points_Scored
0,K. Antonelli,1,Australian GP,No,2,18
1,K. Antonelli,2,Chinese GP,Yes,1,29
2,K. Antonelli,3,Japanese GP,No,1,25
3,K. Antonelli,4,Miami GP,Yes,1,28
4,K. Antonelli,5,Canadian GP,Yes,1,31
5,K. Antonelli,6,Monaco GP,No,1,25
6,K. Antonelli,7,Barcelona GP,No,DNF,0
7,K. Antonelli,8,Austrian GP,No,3,15


## Combine All Races

In [12]:
history['Race_Number'] = pd.to_numeric(history['Race_Number'])
history['Points_Scored'] = pd.to_numeric(history['Points_Scored'], errors='coerce')

played = results[results['Points_Scored'].notna()][history.columns]
all_races = pd.concat([history, played], ignore_index=True)

print(all_races.shape)
print(all_races.groupby('Driver')['Points_Scored'].sum())

(126, 6)
Driver
C. Leclerc       250.0
G. Russell       264.0
K. Antonelli     402.0
L. Hamilton      267.0
L. Norris        264.0
M. Verstappen    304.0
Name: Points_Scored, dtype: float64


## Race Points by Driver

**Note — sprint and fastest-lap points are excluded here.** `Race_Points` below are finish-position points only (25-18-15-12-10-8-6-4-2-1). Sprint points (up to 8 per sprint weekend) and the fastest-lap point (1) are *not* in the simulation pool, so `Race_Points` totals are lower than the `Points_Scored` totals in the tracker — e.g. Antonelli 371 race points vs 402 points scored. The remaining races (Qatar, Abu Dhabi) are not sprint weekends, so this does not affect the remaining-races simulation.


In [13]:
# Finish-position points only — sprint points and the fastest-lap point
# are excluded from the simulation pool (see the note above).
race_pts = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

all_races['Finish_Num'] = pd.to_numeric(all_races['Finish_Position'], errors='coerce')
all_races['Race_Points'] = all_races['Finish_Num'].map(race_pts).fillna(0)

all_races.groupby('Driver')['Race_Points'].agg(['mean', 'sum'])


,mean,sum
Driver,,
C. Leclerc,10.285714,216.0
G. Russell,10.952381,230.0
K. Antonelli,17.666667,371.0
L. Hamilton,11.428571,240.0
L. Norris,11.000000,231.0
M. Verstappen,13.523810,284.0


# Simulation
## Simulate the Remaining Races

In [14]:
rng = np.random.default_rng(2026)
n_sims = 1_000_000
n_left = len(remaining)

pool = {d: all_races.loc[all_races['Driver'] == d, 'Race_Points'].values for d in standings.index}

draws = {d: rng.choice(pool[d], size=(n_sims, n_left)) for d in standings.index}
final = pd.DataFrame({d: current[d] + draws[d].sum(axis=1) for d in standings.index})

final.mean().round(1)

M. Verstappen    331.0
L. Hamilton      289.9
G. Russell       285.9
L. Norris        286.0
C. Leclerc       270.6
dtype: float64

# Findings
## P2 Probabilities

In [15]:
runner_up = final.idxmax(axis=1)
p2_prob = runner_up.value_counts(normalize=True).reindex(standings.index).fillna(0)

(p2_prob * 100).round(2)

M. Verstappen    99.28
L. Hamilton       0.14
G. Russell        0.34
L. Norris         0.23
C. Leclerc        0.00
Name: proportion, dtype: float64

## Expected Final Points and Margins

In [16]:
print(final.quantile([0.05, 0.5, 0.95]).round(0))

sorted_totals = np.sort(final.values, axis=1)[:, ::-1]
margin = sorted_totals[:, 0] - sorted_totals[:, 1]

print(f"Average gap between P2 and the next-best contender: {margin.mean():.1f}")
print(f"Chance that gap is 10 points or less: {(margin <= 10).mean():.1%}")

      M. Verstappen  L. Hamilton  G. Russell  L. Norris  C. Leclerc
0.05          308.0        277.0       266.0      266.0       254.0
0.50          332.0        290.0       286.0      286.0       272.0
0.95          347.0        303.0       307.0      304.0       287.0
Average gap between P2 and the next-best contender: 34.6
Chance that gap is 10 points or less: 5.2%


## Points Needed to Mathematically Clinch P2

In [17]:
max_available = remaining['Max_Points'].sum()
pts = standings['Points']

lead_matrix = pd.DataFrame({b: pts - pts[b] for b in standings.index})
lead_matrix.columns.name = 'vs'

print("Max points still available:", max_available)
lead_matrix

Max points still available: 50


vs,M. Verstappen,L. Hamilton,G. Russell,L. Norris,C. Leclerc
M. Verstappen,0.0,37.0,40.0,40.0,54.0
L. Hamilton,-37.0,0.0,3.0,3.0,17.0
G. Russell,-40.0,-3.0,0.0,0.0,14.0
L. Norris,-40.0,-3.0,0.0,0.0,14.0
C. Leclerc,-54.0,-17.0,-14.0,-14.0,0.0


In [18]:
extra_needed = (max_available - lead_matrix + 1).clip(lower=0)
extra_needed = extra_needed.mask(np.eye(len(extra_needed), dtype=bool))

print(extra_needed)
print()
print("Extra lead needed to clinch P2 (against the closest rival):")
print(extra_needed.max(axis=1))

vs             M. Verstappen  L. Hamilton  G. Russell  L. Norris  C. Leclerc
M. Verstappen            NaN         14.0        11.0       11.0         0.0
L. Hamilton             88.0          NaN        48.0       48.0        34.0
G. Russell              91.0         54.0         NaN       51.0        37.0
L. Norris               91.0         54.0        51.0        NaN        37.0
C. Leclerc             105.0         68.0        65.0       65.0         NaN

Extra lead needed to clinch P2 (against the closest rival):
M. Verstappen     14.0
L. Hamilton       88.0
G. Russell        91.0
L. Norris         91.0
C. Leclerc       105.0
dtype: float64


## Earliest Possible Clinch of P2

In [19]:
cum = {d: draws[d].cumsum(axis=1) + current[d] for d in standings.index}
left_after = remaining['Max_Points'].sum() - remaining['Max_Points'].cumsum().values

clinched_p2 = {}
for d in standings.index:
    best_rival = np.max([cum[r] for r in standings.index if r != d], axis=0)
    clinched_p2[d] = cum[d] > best_rival + left_after

print(left_after)
print({d: clinched_p2[d].shape for d in standings.index})

[25  0]
{'M. Verstappen': (1000000, 2), 'L. Hamilton': (1000000, 2), 'G. Russell': (1000000, 2), 'L. Norris': (1000000, 2), 'C. Leclerc': (1000000, 2)}


In [20]:
race_labels = [f"{name} ({int(num)})" for num, name in zip(remaining['Race_Number'], remaining['Race_Name'])]

by_driver = pd.DataFrame({d: clinched_p2[d].mean(axis=0) for d in standings.index}, index=race_labels)
by_driver['Anyone'] = np.logical_or.reduce([clinched_p2[d] for d in standings.index]).mean(axis=0)

(by_driver * 100).round(2)

,M. Verstappen,L. Hamilton,G. Russell,L. Norris,C. Leclerc,Anyone
Qatar GP (22),79.72,0.00,0.00,0.00,0.0,79.72
Abu Dhabi GP (23),99.02,0.14,0.34,0.23,0.0,99.72
